In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

repo_root = Path.cwd().parent if (Path.cwd() / "analysis_outputs").exists() else Path.cwd()
inventory_path = repo_root / "analysis_outputs" / "blocked_crossings_summary.csv"
yards_path = repo_root / "data" / "rail_yards.geojson"
network_path = repo_root / "data" / "rail_network_lines.geojson"
yard_output_path = repo_root / "analysis_outputs" / "crossings_with_yard_features.csv"
output_path = repo_root / "analysis_outputs" / "crossings_with_yard_and_network_features.csv"

def resolve_column(df: pd.DataFrame, candidates: list[str]) -> str:
    lookup = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lookup:
            return lookup[key]
    raise KeyError(f"None of these columns were found: {candidates}")

def truthy(series: pd.Series) -> pd.Series:
    return series.astype("string").str.upper().isin(["T", "Y", "YES", "1", "P", "TRUE"])

def count_within_radius(points_with_radius: gpd.GeoDataFrame, lines: gpd.GeoDataFrame, id_col: str, out_col: str) -> pd.DataFrame:
    if lines.empty:
        return points_with_radius[[id_col]].drop_duplicates().assign(**{out_col: 0})
    joined = gpd.sjoin(points_with_radius[[id_col, "geometry"]], lines[["geometry"]], how="left", predicate="intersects")
    return joined.groupby(id_col).size().rename(out_col).reset_index()

def sum_within_radius(points_with_radius: gpd.GeoDataFrame, lines: gpd.GeoDataFrame, id_col: str, value_col: str, out_col: str) -> pd.DataFrame:
    if lines.empty:
        return points_with_radius[[id_col]].drop_duplicates().assign(**{out_col: 0.0})
    joined = gpd.sjoin(points_with_radius[[id_col, "geometry"]], lines[["geometry", value_col]], how="left", predicate="intersects")
    return joined.groupby(id_col)[value_col].sum(min_count=1).fillna(0).rename(out_col).reset_index()

def nunique_within_radius(points_with_radius: gpd.GeoDataFrame, lines: gpd.GeoDataFrame, id_col: str, value_col: str, out_col: str) -> pd.DataFrame:
    if lines.empty:
        return points_with_radius[[id_col]].drop_duplicates().assign(**{out_col: 0})
    joined = gpd.sjoin(points_with_radius[[id_col, "geometry"]], lines[["geometry", value_col]], how="left", predicate="intersects")
    return joined.groupby(id_col)[value_col].nunique(dropna=True).rename(out_col).reset_index()

# ------------------------------------------------------------------
# 1. Load the crossing inventory and create point geometry.
# ------------------------------------------------------------------
crossings = pd.read_csv(inventory_path)
crossing_id_col = resolve_column(crossings, ["Crossing ID", "crossing_id"])
lat_col = resolve_column(crossings, ["Latitude", "LATITUDE", "latitude"])
lon_col = resolve_column(crossings, ["Longitude", "LONGITUDE", "longitude"])

crossings_gdf = gpd.GeoDataFrame(
    crossings,
    geometry=gpd.points_from_xy(crossings[lon_col], crossings[lat_col]),
    crs="EPSG:4326",
)

# ------------------------------------------------------------------
# 2. Append yard features and save the intermediate table. 
# The North American Rail Network (NARN) Rail Yards dataset was created in 2024 and was updated on July 21, 2026 from the FRA and is part of the USDOT BTS NTAD.
# Interested in crossing proximity to rail yards within a 5-mile radius and the distance to the nearest yard.
# See https://geodata.bts.gov/datasets/usdot::rail-yards/about
# ------------------------------------------------------------------
yards = gpd.read_file(yards_path)
if yards.crs is None:
    raise ValueError("Rail yards layer has no CRS defined.")

projected_crs = "EPSG:2163"
crossings_p = crossings_gdf.to_crs(projected_crs)
yards_p = yards.to_crs(projected_crs)
yard_radius_m = 8046.72  # 5 miles

yard_nearest = gpd.sjoin_nearest(
    crossings_p,
    yards_p,
    how="left",
    distance_col="dist_to_nearest_yard_m",
)
yard_nearest = yard_nearest.sort_values([crossing_id_col, "dist_to_nearest_yard_m", "YARDNAME"], na_position="last").drop_duplicates(subset=[crossing_id_col], keep="first")
yard_nearest = yard_nearest.drop(columns=["index_right"], errors="ignore")

yard_buffer = crossings_p[[crossing_id_col, "geometry"]].copy()
yard_buffer["geometry"] = yard_buffer.geometry.buffer(yard_radius_m)
yard_counts = count_within_radius(yard_buffer, yards_p, crossing_id_col, "yard_count_within_5_miles")

yard_result = yard_nearest.drop(columns=["geometry"]).merge(yard_counts, on=crossing_id_col, how="left")
yard_result["yard_count_within_5_miles"] = yard_result["yard_count_within_5_miles"].fillna(0).astype(int)
yard_result["dist_to_nearest_yard_miles"] = yard_result["dist_to_nearest_yard_m"] / 1609.344
yard_result = yard_result.drop(columns=["dist_to_nearest_yard_m"], errors="ignore")
yard_result.to_csv(yard_output_path, index=False)

# ------------------------------------------------------------------
# 3. Append rail-network features on top of the yard-enriched table.
# The North American Rail Network (NARN) Rail Lines dataset was created in 2016 and was updated on July 21, 2026 from the FRA and is part of the USDOT BTS NTAD.
# We are interested in the network code, passenger service code, track count, and distance to the nearest line. We also want to count the number of lines within 5 miles and the number of passenger-service lines within 1 mile.
# See https://geodata.bts.gov/datasets/usdot::rail-network-lines/about
# ------------------------------------------------------------------
network = gpd.read_file(network_path)
if network.crs is None:
    raise ValueError("Rail network layer has no CRS defined.")

network_p = network.to_crs(projected_crs).copy()
network_p["PASSNGR"] = network_p["PASSNGR"].astype("string").str.strip().str.upper()
network_p["NET"] = network_p["NET"].astype("string")
network_p["TRACKS"] = pd.to_numeric(network_p["TRACKS"], errors="coerce")
network_p["MILES"] = pd.to_numeric(network_p["MILES"], errors="coerce")
passenger_service_map = {
    "A": "Amtrak",
    "B": "Amtrak & Commuter",
    "C": "Commuter line",
    "D": "Alaska Railroad Passenger Service",
    "E": "Intercity High-Speed Rail & Commuter",
    "I": "Intercity High-Speed Rail",
    "O": "Ontario Northland (Canada Network Only)",
    "R": "Rapid Transit",
    "T": "Tourist, Museum, or Science Passenger Service",
    "V": "Via Rail Canada (Canada Network Only)",
}
network_p["passenger_service_type"] = network_p["PASSNGR"].map(passenger_service_map)
network_p["is_passenger_line"] = network_p["PASSNGR"].isin(passenger_service_map)
network_p["is_yard_line"] = network_p["NET"].str.upper().eq("Y") | network_p["YARDNAME"].notna()
network_p["is_multi_track_line"] = network_p["TRACKS"].fillna(0) >= 2
network_code_map = {
    "M": {"name": "Main Line", "description": "Primary mainline track carrying through-freight and/or passenger operations."},
    "I": {"name": "Major Industrial Lead", "description": "Major industrial track or lead feeding industrial customers."},
    "O": {"name": "Other Track", "description": "Minor industrial leads, spurs, and secondary localized switching tracks."},
    "S": {"name": "Passing Siding", "description": "Passing tracks and auxiliary sidings alongside mainlines."},
    "Y": {"name": "Yard Tracks", "description": "Tracks located inside terminal, classification, or switching yards."},
    "X": {"name": "Out of Service Line", "description": "Active right-of-way, but currently out of service or inactive."},
    "A": {"name": "Abandoned Line", "description": "Officially abandoned rail line segment."},
    "R": {"name": "Removed Line", "description": "Abandoned track where physical rails/ties have been removed."},
    "F": {"name": "Rail Ferry Connection", "description": "Maritime rail barge/ferry transfer link."},
}
network_code_lookup = pd.DataFrame(
    [
        {"Code": code, "Value Name": meta["name"], "Description": meta["description"]}
        for code, meta in network_code_map.items()
    ]
)
print("Network code legend")
print(network_code_lookup.to_string(index=False))
passenger_service_lookup = pd.DataFrame(
    [
        {"Code": code, "Value Name": name}
        for code, name in passenger_service_map.items()
    ]
)
print()
print("Passenger service legend")
print(passenger_service_lookup.to_string(index=False))

network_radius_m = 8046.72
passenger_radius_m = 1609.344

nearest_network = gpd.sjoin_nearest(
    crossings_p,
    network_p[[
        "geometry",
        "NET",
        "TRACKS",
        "MILES",
        "YARDNAME",
        "RROWNER1",
        "DIVISION",
        "SUBDIV",
        "BRANCH",
        "is_passenger_line",
        "is_yard_line",
        "is_multi_track_line",
    ]],
    how="left",
    distance_col="dist_to_nearest_network_line_m",
)
nearest_network = nearest_network.sort_values([crossing_id_col, "dist_to_nearest_network_line_m", "NET", "RROWNER1"], na_position="last").drop_duplicates(subset=[crossing_id_col], keep="first")
nearest_network = nearest_network.drop(columns=["index_right"], errors="ignore")

passenger_nearest = gpd.sjoin_nearest(
    crossings_p,
    network_p.loc[network_p["is_passenger_line"], ["geometry", "NET", "PASSNGR", "passenger_service_type", "TRACKS", "MILES", "RROWNER1"]],
    how="left",
    distance_col="dist_to_nearest_passenger_line_m",
)
passenger_nearest = passenger_nearest.sort_values([crossing_id_col, "dist_to_nearest_passenger_line_m", "NET", "RROWNER1"], na_position="last").drop_duplicates(subset=[crossing_id_col], keep="first")
passenger_nearest = passenger_nearest[[crossing_id_col, "dist_to_nearest_passenger_line_m", "PASSNGR", "passenger_service_type"]]

network_buffer = crossings_p[[crossing_id_col, "geometry"]].copy()
network_buffer["geometry"] = network_buffer.geometry.buffer(network_radius_m)
passenger_buffer = crossings_p[[crossing_id_col, "geometry"]].copy()
passenger_buffer["geometry"] = passenger_buffer.geometry.buffer(passenger_radius_m)

all_lines = count_within_radius(network_buffer, network_p, crossing_id_col, "line_count_within_5_miles")
passenger_lines = count_within_radius(passenger_buffer, network_p.loc[network_p["is_passenger_line"]], crossing_id_col, "passenger_service_segments_within_1_mile")
passenger_miles = sum_within_radius(passenger_buffer, network_p.loc[network_p["is_passenger_line"]], crossing_id_col, "MILES", "passenger_service_miles_within_1_mile")
passenger_types = nunique_within_radius(network_buffer, network_p.loc[network_p["is_passenger_line"]], crossing_id_col, "passenger_service_type", "passenger_service_types_within_5_miles")

yard_features = yard_result[[
    crossing_id_col,
    "OBJECTID",
    "YARDNAME",
    "RROWNER1_NAME",
    "yard_count_within_5_miles",
    "dist_to_nearest_yard_miles",
]].rename(columns={
    "OBJECTID": "nearest_yard_objectid",
    "YARDNAME": "nearest_yard_name",
    "RROWNER1_NAME": "nearest_yard_owner_name",
    "yard_count_within_5_miles": "yards_within_5_miles",
    "dist_to_nearest_yard_miles": "distance_to_nearest_yard_miles",
})

result = nearest_network.drop(columns=["geometry"]).merge(passenger_nearest, on=crossing_id_col, how="left")
result = result.merge(yard_features, on=crossing_id_col, how="left")
result = result.merge(all_lines, on=crossing_id_col, how="left")
result = result.merge(passenger_lines, on=crossing_id_col, how="left")
result = result.merge(passenger_miles, on=crossing_id_col, how="left")
result = result.merge(passenger_types, on=crossing_id_col, how="left")

result = result.rename(columns={
    "NET": "nearest_network_code",
    "PASSNGR": "nearest_passenger_service_code",
    "passenger_service_type": "nearest_passenger_service_type",
    "TRACKS": "nearest_track_count",
    "MILES": "nearest_segment_miles",
    "YARDNAME": "nearest_line_yardname",
    "RROWNER1": "nearest_owner",
    "DIVISION": "nearest_division",
    "SUBDIV": "nearest_subdivision",
    "BRANCH": "nearest_branch",
    "is_passenger_line": "nearest_is_passenger_line",
    "is_yard_line": "nearest_is_yard_line",
    "is_multi_track_line": "nearest_is_multi_track_line",
})
result["nearest_network_code_name"] = result["nearest_network_code"].map(lambda value: network_code_map.get(value, {}).get("name"))
result["nearest_network_code_description"] = result["nearest_network_code"].map(lambda value: network_code_map.get(value, {}).get("description"))

result["dist_to_nearest_network_line_miles"] = result["dist_to_nearest_network_line_m"] / 1609.344
result["dist_to_nearest_passenger_line_miles"] = result["dist_to_nearest_passenger_line_m"] / 1609.344
result["nearest_is_passenger_line"] = result["nearest_is_passenger_line"].fillna(False).astype(int)
result["nearest_is_yard_line"] = result["nearest_is_yard_line"].fillna(False).astype(int)
result["nearest_is_multi_track_line"] = result["nearest_is_multi_track_line"].fillna(False).astype(int)
for column in ["line_count_within_5_miles", "passenger_service_segments_within_1_mile", "yards_within_5_miles", "passenger_service_types_within_5_miles"]:
    result[column] = result[column].fillna(0).astype(int)
result["passenger_service_miles_within_1_mile"] = result["passenger_service_miles_within_1_mile"].fillna(0.0)

result = result.drop(columns=["dist_to_nearest_network_line_m", "dist_to_nearest_passenger_line_m"], errors="ignore")
result.to_csv(output_path, index=False)

print(f"Saved yard table: {yard_output_path}")
print(f"Saved network table: {output_path}")
print(f"Rows: {len(result):,}")
print(f"Unique crossings: {result[crossing_id_col].nunique():,}")
print()
print("Nearest network code legend")
print(network_code_lookup.to_string(index=False))
print(result[[crossing_id_col, "distance_to_nearest_yard_miles", "nearest_yard_name", "nearest_network_code", "nearest_network_code_name", "nearest_passenger_service_code", "nearest_passenger_service_type", "dist_to_nearest_network_line_miles", "dist_to_nearest_passenger_line_miles"]].head(10).to_string(index=False))


In [ ]:
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().parent if (Path.cwd() / "analysis_outputs").exists() else Path.cwd()
static_path = repo_root / "analysis_outputs" / "crossings_with_yard_and_network_features.csv"
events_path = repo_root / "analysis_outputs" / "blocked_events_joined_to_inventory.csv"
inventory_path = repo_root / "data" / "Crossing_Inventory_Data_(Form_71)_-_Current_20260707.csv"
output_path = repo_root / "analysis_outputs" / "crossing_month_panel.csv"

def resolve_column(df: pd.DataFrame, candidates: list[str]) -> str:
    lookup = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lookup:
            return lookup[key]
    raise KeyError(f"None of these columns were found: {candidates}")

def duration_to_minutes(series: pd.Series) -> pd.Series:
    mapping = {
        "0-15 minutes": 7.5,
        "16-30 minutes": 23.0,
        "31-60 minutes": 45.5,
        "1-2 hours": 90.0,
        "2-6 hours": 240.0,
        "6-12 hours": 540.0,
        "12-24 hours": 1080.0,
        "More than one day": 1440.0,
    }
    cleaned = series.astype("string").str.strip()
    return cleaned.map(mapping)

static = pd.read_csv(static_path)
events = pd.read_csv(events_path, low_memory=False)

crossing_id_col = resolve_column(static, ["Crossing ID", "crossing_id"])
event_id_col = resolve_column(events, ["Crossing ID", "crossing_id"])
date_col = resolve_column(events, ["Date/Time", "Date Time", "date_time"])
duration_col = resolve_column(events, ["Duration", "duration"])
reason_col = resolve_column(events, ["Reason", "reason"])
impacts_col = resolve_column(events, ["Immediate Impacts", "Immediate impacts", "immediate_impacts"])

events[date_col] = pd.to_datetime(events[date_col], errors="coerce")
if events[date_col].isna().any():
    raise ValueError("Some blocked event Date/Time values could not be parsed.")

events["panel_month"] = events[date_col].dt.to_period("M").dt.to_timestamp()
events["duration_minutes_est"] = duration_to_minutes(events[duration_col])
events["is_severe_block"] = events["duration_minutes_est"].fillna(0) >= 120
reason = events[reason_col].astype("string").str.strip().str.lower()
events["is_stationary_train_reason"] = reason.eq("a stationary train")
events["is_moving_train_reason"] = reason.eq("a moving train")
events["is_signal_false_positive_reason"] = reason.str.contains("lights and/or gates were activated", na=False)
impacts = events[impacts_col].astype("string")
events["is_first_responder_impact"] = impacts.str.contains("first responders", case=False, na=False)
events["is_pedestrian_impact"] = impacts.str.contains("pedestrians", case=False, na=False)

monthly = events.groupby([event_id_col, "panel_month"], as_index=False).agg(
    blocked_event_count_month=(event_id_col, "size"),
    blocked_duration_mean_minutes_month=("duration_minutes_est", "mean"),
    blocked_duration_max_minutes_month=("duration_minutes_est", "max"),
    severe_block_count_month=("is_severe_block", "sum"),
    stationary_train_count_month=("is_stationary_train_reason", "sum"),
    moving_train_count_month=("is_moving_train_reason", "sum"),
    signal_false_positive_count_month=("is_signal_false_positive_reason", "sum"),
    first_responder_impact_count_month=("is_first_responder_impact", "sum"),
    pedestrian_impact_count_month=("is_pedestrian_impact", "sum"),
)
monthly = monthly.rename(columns={event_id_col: crossing_id_col})

months = pd.period_range(events["panel_month"].min(), events["panel_month"].max(), freq="M").to_timestamp()
crossings = static[[crossing_id_col]].drop_duplicates().assign(_join_key=1)
month_frame = pd.DataFrame({"panel_month": months, "_join_key": 1})
panel = crossings.merge(month_frame, on="_join_key", how="left").drop(columns="_join_key")
panel = panel.merge(monthly, on=[crossing_id_col, "panel_month"], how="left")
panel = panel.merge(static, on=crossing_id_col, how="left")

# Pull a small set of design/context fields from the current Form 71 snapshot.
# These are mostly static crossing attributes, so they belong on every monthly row.
inventory = pd.read_csv(inventory_path, low_memory=False)
inventory_id_col = resolve_column(inventory, ["Crossing ID", "crossing_id"])
revision_col = resolve_column(inventory, ["Revision Date", "revision date"])
inventory[revision_col] = pd.to_datetime(inventory[revision_col], errors="coerce")
inventory = inventory.sort_values([inventory_id_col, revision_col]).drop_duplicates(inventory_id_col, keep="last")

inventory_feature_specs = {
    "inv_smallest_crossing_angle": ["Smallest Crossing Angle"],
    "inv_road_at_crossing": ["Road At Crossing"],
    "inv_road_at_crossing_type": ["Road At Crossing Type"],
    "inv_gate_configuration": ["Gate Configuration"],
    "inv_gate_configuration_type": ["Gate Configuration Type"],
    "inv_track_signaled": ["Track Signaled"],
    "inv_number_siding_tracks": ["Number Of Siding Tracks"],
    "inv_signs_or_signals": ["Signs Or Signals"],
    "inv_warning_device_code": ["Warning Device Code"],
    "inv_number_crossbuck_assemblies": ["Number Crossbuck Assemblies"],
    "inv_count_roadway_gate_arms": ["Count Roadway Gate Arms"],
    "inv_highway_traffic_signals": ["Highway Traffic Signals"],
    "inv_highway_traffic_signal_interconnection": ["Highway Traffic Signal Interconnection"],
    "inv_highway_traffic_signal_preemption": ["Highway Traffic Signal Preemption"],
    "inv_crossing_surface_description_1": ["Crossing Surface Description 1"],
    "inv_crossing_surface_description_2": ["Crossing Surface Description 2"],
    "inv_crossing_surface_description_3": ["Crossing Surface Description 3"],
    "inv_other_crossing_surface": ["Other Crossing Surface"],
    "inv_reporting_railroad_class": ["Reporting Railroad Class"],
}
inventory_feature_cols = [inventory_id_col]
inventory_feature_rename: dict[str, str] = {}
for output_name, candidates in inventory_feature_specs.items():
    try:
        source_name = resolve_column(inventory, candidates)
    except KeyError:
        continue
    inventory_feature_cols.append(source_name)
    inventory_feature_rename[source_name] = output_name

inventory_features = inventory[inventory_feature_cols].rename(columns=inventory_feature_rename)
inventory_features = inventory_features.rename(columns={inventory_id_col: crossing_id_col})
panel = panel.merge(inventory_features, on=crossing_id_col, how="left")

panel["panel_year"] = panel["panel_month"].dt.year
panel["panel_month_num"] = panel["panel_month"].dt.month
panel["panel_month_index"] = panel["panel_year"] * 12 + panel["panel_month_num"]
panel = panel.sort_values([crossing_id_col, "panel_month"]).reset_index(drop=True)
panel["month_index_from_start"] = panel.groupby(crossing_id_col).cumcount()
panel["is_blocked_month"] = (panel["blocked_event_count_month"].fillna(0) > 0).astype(int)

count_cols = [
    "blocked_event_count_month",
    "severe_block_count_month",
    "stationary_train_count_month",
    "moving_train_count_month",
    "signal_false_positive_count_month",
    "first_responder_impact_count_month",
    "pedestrian_impact_count_month",
]
panel[count_cols] = panel[count_cols].fillna(0).astype(int)
panel[["blocked_duration_mean_minutes_month", "blocked_duration_max_minutes_month"]] = panel[["blocked_duration_mean_minutes_month", "blocked_duration_max_minutes_month"]].fillna(0)

for lag in [1, 3, 6, 12]:
    panel[f"blocked_event_count_lag{lag}"] = panel.groupby(crossing_id_col)["blocked_event_count_month"].shift(lag).fillna(0).astype(int)

for window in [3, 6, 12]:
    panel[f"blocked_event_count_roll{window}"] = panel.groupby(crossing_id_col)["blocked_event_count_month"].transform(
        lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
    ).fillna(0).astype(int)

def months_since_last_block(counts: pd.Series) -> pd.Series:
    values = counts.fillna(0).to_numpy()
    out: list[object] = []
    last_block_idx: int | None = None
    for idx, value in enumerate(values):
        out.append(pd.NA if last_block_idx is None else idx - last_block_idx)
        if value > 0:
            last_block_idx = idx
    return pd.Series(out, index=counts.index, dtype="Int64")

panel["months_since_last_block"] = panel.groupby(crossing_id_col)["blocked_event_count_month"].transform(months_since_last_block)
panel.to_csv(output_path, index=False)

print(f"Saved panel: {output_path}")
print(f"Rows: {len(panel):,}")
print(f"Crossings: {panel[crossing_id_col].nunique():,}")
print(panel[[crossing_id_col, "panel_month", "blocked_event_count_month", "blocked_event_count_lag1", "blocked_event_count_roll3", "months_since_last_block", "inv_smallest_crossing_angle", "inv_gate_configuration", "inv_reporting_railroad_class"]].head(12).to_string(index=False))
